[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philmui/worldmodels/blob/main/gait/skeleton-jepa/gavd2/01-bulk-download-youtube.ipynb)

# Part 1: Bulk download every referenced video

Notebook 00 gave us a manifest with one row per sequence and, crucially, the
YouTube video id each sequence comes from. This notebook turns that list of ids
into actual video files on disk. Once the videos are cached, notebook 02 can run
pose estimation over them without touching the network again.

The key move here is deduplication. Many sequences share a single video, so the
manifest's 374 sequences point at far fewer unique clips. Downloading each unique
video once, and skipping anything already cached, keeps this polite to YouTube
and fast to resume. If your connection drops halfway through, just run the
notebook again and it picks up where it left off.

We download using the alexpose `YouTubeHandler` when its `ambient` package is
importable, which wraps `yt-dlp` and names each file `<video_id>.mp4`. When the
private repo is not available, for example in a fresh Colab, we fall back to
calling `yt-dlp` directly with the same output naming, so the rest of the series
does not care which path fetched the file.

## Run this locally or in Google Colab

In Colab, click the badge and run top to bottom. On your laptop, from this
`gavd2/` folder:

```bash
uv sync --extra real     # the real path needs yt-dlp for downloading
uv run jupyter lab 01-bulk-download-youtube.ipynb
```

With `SMOKE_TEST = True`, this notebook downloads nothing. It builds a tiny
synthetic manifest and simulates the download bookkeeping so you can see exactly
what the real run would do. With `SMOKE_TEST = False` (the value these committed
notebooks ship with) it reads the real manifest from notebook 00 and downloads the
unique videos into your cache. The real run can take a while and use real disk,
since it fetches every referenced clip by default. You can set `MAX_VIDEOS` to a
small number for a quick trial.

## What this step does

The figure shows the dedup-and-download flow: the manifest's many sequence rows
collapse to a set of unique video ids, each id is downloaded once (or skipped if
already cached), and we record what succeeded in a small report.

![Dedup and bulk download](images/bulk-download.svg)

*Many sequences map to fewer unique videos; we download each once, resume cleanly, and write a per-video report.*

## Colab setup

In [ ]:
# Colab setup and local .env loading.
import importlib.util, shutil, subprocess, sys

_import_name = {"scikit-learn": "sklearn", "opencv-python": "cv2",
                "yt-dlp[default]": "yt_dlp_ejs", "python-dotenv": "dotenv"}

def _ensure(pkgs):
    """pip install any packages whose import is not already available."""
    missing = [p for p in pkgs if importlib.util.find_spec(_import_name.get(p, p)) is None]
    if missing:
        print("Installing:", " ".join(missing))
        if importlib.util.find_spec("pip") is not None:
            cmd = [sys.executable, "-m", "pip", "install", "-q"] + missing
        elif shutil.which("uv"):
            cmd = ["uv", "pip", "install", "-q"] + missing
        else:
            raise RuntimeError("Missing packages but neither pip nor uv is available: " + ", ".join(missing))
        subprocess.check_call(cmd)
    return missing

# Core packages, always needed (smoke mode uses only these).
_ensure(["numpy", "pandas", "matplotlib", "python-dotenv", "tqdm"])

# yt-dlp[default] is the REAL-path YouTube extractor and is only installed lazily
# inside download_one below, so a SMOKE run in a venv without pip stays green.

# Credentials, keys, and local paths via load_dotenv(find_dotenv()).
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())
print("Loaded environment via load_dotenv(find_dotenv()).")

ALEXPOSE_REPO = os.getenv("ALEXPOSE_REPO")
GAVD_DATA_DIR = os.getenv("GAVD_DATA_DIR")
YOUTUBE_CACHE_DIR = os.getenv("YOUTUBE_CACHE_DIR")
GAVD_CACHE_DIR = os.getenv("GAVD_CACHE_DIR")
# iteration-2 keys, harmless if unset; nb01 reads none of them, but importing the
# same names as nb00 keeps every notebook's env footprint identical.
EXP4_DATA_DIR = os.getenv("EXP4_DATA_DIR")
EXP5_FEATURES_PKL = os.getenv("EXP5_FEATURES_PKL")
if ALEXPOSE_REPO and os.path.isdir(ALEXPOSE_REPO) and ALEXPOSE_REPO not in sys.path:
    sys.path.insert(0, ALEXPOSE_REPO)
print("Setup complete.")

## Configuration

`MAX_VIDEOS` controls how many unique videos the real path downloads. It is
`None` by default, which means download every unique video the manifest
references. Set it to a small integer like 3 for a quick real trial before
committing to the full pull.

In [ ]:
from pathlib import Path
import hashlib

CONFIG = {
    "SMOKE_TEST": False,         # True -> no network, simulate the bookkeeping.
    "CACHE_DIR": Path(GAVD_CACHE_DIR) if GAVD_CACHE_DIR else Path.cwd() / "cache",
    "YOUTUBE_DIR": Path(YOUTUBE_CACHE_DIR) if YOUTUBE_CACHE_DIR else Path.home() / "dev" / "alexpose" / "data" / "youtube",
    "MAX_VIDEOS": None,          # None -> download every unique video. Set an int for a quick trial.
    "QUALITY": "bv*[height<=720][ext=mp4]+ba[ext=m4a]/b[height<=720][ext=mp4]/best[height<=720]",
    "MIN_VALID_VIDEO_BYTES": 64 * 1024,
    "YT_DLP_JS_RUNTIMES": {"node": {}},      # Use {"deno": {}} if you install deno instead.
    "YT_DLP_REMOTE_COMPONENTS": [],          # yt-dlp[default] installs local EJS scripts.
    "YT_DLP_COOKIES_FROM_BROWSER": None,     # Example for age/sign-in friction: ("chrome",)
    # When True, use the exploratory first-N labelling cache namespace so this run does
    # not overwrite the locked run's artifacts. Notebook 00 defines the semantics.
    "EXPLORATORY_FIRST_N": False,
}
# Cache namespace: "" for the locked run, "_firstN" for the exploratory run. Every
# artifact filename in the series carries this suffix so the two never mix.
CONFIG["CACHE_NS"] = "_firstN" if CONFIG.get("EXPLORATORY_FIRST_N") else ""
CONFIG["CACHE_DIR"].mkdir(parents=True, exist_ok=True)
CONFIG["YOUTUBE_DIR"].mkdir(parents=True, exist_ok=True)

# Shared helpers threaded through every notebook in the series (mirrors nb00).
# Canonicalize condition spellings so the cerebral-palsy class never silently drops.
CANONICAL_COND = {"cerebral palsy": "cerebralpalsy"}
def canon_cond(c):
    c = str(c).strip().lower()
    return CANONICAL_COND.get(c, c.replace(" ", ""))

# Canonical label -> on-disk GAVD folder name (inverse of canon for the space case).
COND_TO_FOLDER = {"cerebralpalsy": "cerebral palsy"}

# Deterministic fingerprint of the locked 68 ids, stamped onto every cache artifact.
def canonical_id_hash(ids):
    return hashlib.sha1("\n".join(sorted(map(str, ids))).encode()).hexdigest()[:12]

print("CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k:18s} = {v}")

## Load the manifest and dedup to unique videos

We read the manifest written by notebook 00. If it is missing, for example
because you opened this notebook first, we fall back to a small synthetic manifest
so the notebook still runs. Then we collapse the sequence rows down to the set of
unique video ids, which is the true download list.

In [ ]:
import pandas as pd

ns = CONFIG["CACHE_NS"]
manifest_path = CONFIG["CACHE_DIR"] / f"manifest{ns}.csv"

def synthetic_manifest():
    counts = {"abnormal": 6, "myopathic": 3, "parkinsons": 2, "stroke": 2, "normal": 2}
    rows, vid = [], 0
    for cond, n in counts.items():
        for i in range(n):
            vid_id = f"vid{vid // 2:05d}"; vid += 1
            rows.append({"seq": f"cl{cond[:3]}{i:04d}xxxxxxxxxxxxxxxx", "condition": cond,
                         "video_id": vid_id, "url": f"https://www.youtube.com/watch?v={vid_id}",
                         "start_frame": 100, "end_frame": 139, "num_frames": 40,
                         "has_bbox": True, "is_labeled": cond != "abnormal"})
    return pd.DataFrame(rows)

if CONFIG["SMOKE_TEST"]:
    manifest = synthetic_manifest()
    print("SMOKE mode: using a synthetic manifest (no real download).")
elif manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    print(f"Loaded manifest from {manifest_path}")
else:
    manifest = synthetic_manifest()
    print(f"No manifest{ns}.csv found; run notebook 00 first. Falling back to synthetic.")

# Unique videos, carrying condition + is_labeled so a capped or interrupted run can
# fetch the labelled-68 videos first. Sorting by is_labeled DESC before any head()
# means MAX_VIDEOS=k still yields the k that matter most for the probe.
unique_videos = (manifest.drop_duplicates("video_id")
                        [["video_id", "url", "condition", "is_labeled"]])
unique_videos = (unique_videos.sort_values("is_labeled", ascending=False)
                              .reset_index(drop=True))
if CONFIG["MAX_VIDEOS"] is not None:
    unique_videos = unique_videos.head(CONFIG["MAX_VIDEOS"])

print(f"\n{len(manifest)} sequences -> {len(unique_videos)} unique videos to fetch")
unique_videos.head()

## A download function with a clean fallback

The function below fetches one video id. In real mode it validates the expected
`<video_id>.mp4` cache file first; a playable cached video is skipped, while a
bad or partial cache file is removed and downloaded again. New downloads are
validated before they are reported as usable. In smoke mode it does no network
work at all and simply reports the file as simulated, so the bookkeeping
downstream looks identical without any bytes moving.

In [ ]:
import json, shutil, subprocess

def video_file_status(path, min_bytes=64 * 1024):
    """Return (valid, size_mb, note) for a cached or newly downloaded video."""
    path = Path(path)
    if not path.exists():
        return False, 0.0, "missing"
    if not path.is_file():
        return False, 0.0, "not a regular file"

    size = path.stat().st_size
    size_mb = size / 1e6
    if size < min_bytes:
        return False, size_mb, f"too small ({size} bytes)"

    ffprobe = shutil.which("ffprobe")
    if ffprobe:
        cmd = [
            ffprobe, "-v", "error", "-select_streams", "v:0",
            "-show_entries", "stream=codec_type,duration,nb_frames:format=duration",
            "-of", "json", str(path),
        ]
        try:
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=20)
        except Exception as e:
            return False, size_mb, f"ffprobe failed: {e}"
        if proc.returncode != 0:
            msg = (proc.stderr or proc.stdout or "ffprobe returned an error").strip().splitlines()
            return False, size_mb, msg[0][:200] if msg else "ffprobe returned an error"
        try:
            meta = json.loads(proc.stdout or "{}")
        except json.JSONDecodeError as e:
            return False, size_mb, f"ffprobe JSON parse failed: {e}"

        streams = meta.get("streams") or []
        duration_values = [
            (meta.get("format") or {}).get("duration"),
            *[stream.get("duration") for stream in streams],
        ]
        durations = []
        for value in duration_values:
            try:
                if value is not None:
                    durations.append(float(value))
            except (TypeError, ValueError):
                pass
        has_video = any(stream.get("codec_type") == "video" for stream in streams)
        duration = max(durations, default=0.0)
        if has_video and duration > 0:
            return True, size_mb, f"valid video ({duration:.1f}s)"
        return False, size_mb, "ffprobe found no playable video stream"

    try:
        import cv2
    except Exception:
        return True, size_mb, "basic file check only (ffprobe/cv2 unavailable)"

    cap = cv2.VideoCapture(str(path))
    try:
        opened = cap.isOpened()
        width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
        height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
        frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        ok = bool(opened and width > 0 and height > 0 and frames > 0)
        return ok, size_mb, "valid video via cv2" if ok else "cv2 could not read video frames"
    finally:
        cap.release()

def remove_invalid_download(video_id, youtube_dir, dest):
    """Remove stale/partial files for one video id so a retry starts cleanly."""
    candidates = [dest]
    candidates.extend(youtube_dir.glob(f"{video_id}.*.part"))
    candidates.extend(youtube_dir.glob(f"{video_id}.*.ytdl"))
    for candidate in candidates:
        try:
            if candidate.exists():
                candidate.unlink()
        except OSError:
            pass

def download_one(video_id, url, youtube_dir, quality, smoke):
    """Return a dict describing the outcome for one video id."""
    dest = youtube_dir / f"{video_id}.mp4"
    min_bytes = CONFIG.get("MIN_VALID_VIDEO_BYTES", 64 * 1024)
    if smoke:
        return {"video_id": video_id, "cached": False, "ok": True, "valid": True,
                "size_mb": 0.0, "path": "", "note": "smoke (simulated)"}

    valid, size_mb, validation_note = video_file_status(dest, min_bytes)
    if valid:
        return {"video_id": video_id, "cached": True, "ok": True, "valid": True,
                "size_mb": size_mb, "path": str(dest), "note": f"already cached; {validation_note}"}

    cache_note = f"not cached ({validation_note})"
    if dest.exists():
        cache_note = f"cached file rejected: {validation_note}"
        remove_invalid_download(video_id, youtube_dir, dest)

    # Try the alexpose handler first.
    try:
        from ambient.video.youtube_handler import YouTubeHandler
        handler = YouTubeHandler(download_dir=youtube_dir, quality=quality)
        got = handler.get_or_download_video(url)
        got_path = Path(got) if got else dest
        if got_path.exists() and got_path != dest and got_path.suffix.lower() == ".mp4" and not dest.exists():
            shutil.copy2(got_path, dest)
            got_path = dest
        valid, size_mb, validation_note = video_file_status(got_path, min_bytes)
        if valid:
            return {"video_id": video_id, "cached": False, "ok": True, "valid": True,
                    "size_mb": size_mb, "path": str(got_path), "note": f"ambient handler; {validation_note}"}
        note = f"ambient produced invalid file: {validation_note}"
        if got_path == dest:
            remove_invalid_download(video_id, youtube_dir, dest)
    except Exception as e:
        note = f"ambient failed: {e}"
    else:
        note = note or "ambient returned nothing"
    # Fall back to yt-dlp directly.
    try:
        import yt_dlp
        opts = {
            "format": quality,
            "outtmpl": str(youtube_dir / "%(id)s.%(ext)s"),
            "merge_output_format": "mp4",
            "noplaylist": True,
            "quiet": True,
            "retries": 10,
            "fragment_retries": 10,
            "extractor_retries": 3,
            "retry_sleep": {"http": "linear=1:8:1", "fragment": "exp=1:16"},
            "sleep_interval_requests": 0.75,
            "sleep_interval": 2,
            "max_sleep_interval": 8,
            "js_runtimes": CONFIG.get("YT_DLP_JS_RUNTIMES", {"node": {}}),
            "remote_components": CONFIG.get("YT_DLP_REMOTE_COMPONENTS", []),
        }
        if CONFIG.get("YT_DLP_COOKIES_FROM_BROWSER"):
            opts["cookiesfrombrowser"] = CONFIG["YT_DLP_COOKIES_FROM_BROWSER"]
        with yt_dlp.YoutubeDL(opts) as ydl:
            ydl.download([url])
        valid, size_mb, validation_note = video_file_status(dest, min_bytes)
        if valid:
            return {"video_id": video_id, "cached": False, "ok": True, "valid": True,
                    "size_mb": size_mb, "path": str(dest), "note": f"yt-dlp fallback; {validation_note}"}
        remove_invalid_download(video_id, youtube_dir, dest)
        note = f"yt-dlp produced invalid file after {cache_note}: {validation_note}"
    except Exception as e:
        note = f"yt-dlp failed after {cache_note}: {e}"
    return {"video_id": video_id, "cached": False, "ok": False, "valid": False,
            "size_mb": 0.0, "path": str(dest), "note": note}

print("download_one defined.")

## Run the downloads

Now we loop over the unique videos and download each one, showing a progress bar.
Because the loop skips anything already cached, running this cell twice does far
less work the second time. We collect the per-video outcomes so we can report and
save them.

In [ ]:
from tqdm.auto import tqdm

results = []
for _, row in tqdm(unique_videos.iterrows(), total=len(unique_videos), desc="videos"):
    results.append(download_one(row["video_id"], row["url"],
                                CONFIG["YOUTUBE_DIR"], CONFIG["QUALITY"], CONFIG["SMOKE_TEST"]))

report = pd.DataFrame(results)
n_ok = int(report["ok"].sum())
n_cached = int(report["cached"].sum())
n_fresh = int(((~report["cached"]) & report["ok"]).sum())
print(f"\nReady {n_ok}/{len(report)} videos "
      f"({n_cached} valid cache hits, {n_fresh} fresh successes).")
if (~report["ok"]).any():
    print("Failed video ids (dead links or region blocks are common):")
    for _, failed in report.loc[~report["ok"], ["video_id", "note"]].head(10).iterrows():
        print(f"  {failed['video_id']}: {failed['note']}")

# Labelled-video coverage: how many videos that back one of the 68 labelled sequences
# actually landed on disk. A shortfall here is what tanks the frozen-probe evaluation,
# so we report it prominently before the pretty chart.
lab = manifest[manifest["is_labeled"]][["video_id"]].drop_duplicates()
if len(lab):
    ready = report.merge(lab, on="video_id")
    n_lab_ok = int(ready["ok"].sum())
    print(f"\nLabelled videos ready: {n_lab_ok}/{len(lab)}")
    if n_lab_ok < len(lab):
        print(f"WARNING: {len(lab) - n_lab_ok} labelled-set videos are missing; the "
              f"frozen-probe evaluation will be lossy until these are recovered.")

## Report: how much did we get?

The chart summarizes the outcome of the download pass. In a real run some links
will be dead or region-blocked, and that is expected. The proposal plans for it
by windowing sequences with overlap and folding in extra clips, so a few missing
videos do not sink the pretraining pool.

In [ ]:
import matplotlib.pyplot as plt

status = {
    "cached": int((report["cached"] & report["ok"]).sum()),
    "downloaded": int((~report["cached"] & report["ok"]).sum()),
    "failed": int((~report["ok"]).sum()),
}
colors = {"cached": "#22c55e", "downloaded": "#3b82f6", "failed": "#ef4444"}
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(list(status.keys()), list(status.values()),
       color=[colors[k] for k in status])
for i, (k, v) in enumerate(status.items()):
    ax.text(i, v + 0.05, str(v), ha="center", fontsize=11)
ax.set_ylabel("number of videos")
ax.set_title(f"Bulk download outcome ({len(report)} unique videos)")
plt.tight_layout(); plt.show()

total_mb = report["size_mb"].sum()
print(f"Total cached video size touched this run: {total_mb:.1f} MB")

## Save the download report

We write the per-video report to the cache. Notebook 02 reads it to know which
videos are actually on disk, so it only tries to extract skeletons for sequences
whose video downloaded successfully.

In [ ]:
ns = CONFIG["CACHE_NS"]

# Merge condition + is_labeled onto the per-video report so notebook 02 (and any
# auditor) can filter directly on the columns it cares about, without re-joining
# on the manifest. We keep the first row per video for these two fields.
video_meta = (manifest.drop_duplicates("video_id")
                     [["video_id", "condition", "is_labeled"]])
report = report.merge(video_meta, on="video_id", how="left")

report_path = CONFIG["CACHE_DIR"] / f"download_report{ns}.csv"
report.to_csv(report_path, index=False)
print(f"Wrote download report to {report_path}")

# Stamp the canonical id fingerprint if notebook 00 left one, so this artifact is
# traceable back to the exact locked 68 ids the manifest was built from.
hash_path = CONFIG["CACHE_DIR"] / f"canonical_id_hash{ns}.txt"
if hash_path.exists():
    print(f"Canonical id hash: {hash_path.read_text().strip()}")

print(report.head())

## The motion we are downloading for

Downloading is bookkeeping, but the point of every one of those files is the
walking motion inside it. To keep that in view, we animate a walking skeleton
inline, the same kind of movement notebook 02 will extract from these videos. In
this notebook the animation is a small synthetic walker so it needs no video.

In [ ]:
# Inline animation helper: animate_skeleton (self-contained, works in Jupyter and Colab).
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

EDGES = [
    (0,1),(1,2),(2,3),(0,4),(4,5),(5,6),(0,9),(0,10),(9,10),
    (11,12),(11,23),(12,24),(23,24),
    (11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (23,25),(25,27),(27,29),(27,31),(29,31),
    (24,26),(26,28),(28,30),(28,32),(30,32),
]

def synthesize_walking_skeleton(T=16, seed=0):
    """A plausible synthetic (T, 33, 3) walking skeleton for the preview animation."""
    rng = np.random.RandomState(seed)
    base = np.zeros((33, 3), dtype=np.float32)
    # Full (x, y) layout for all 33 landmarks, laid out as a person seen head-on.
    # y grows downward (head near 0.16, feet near 0.97). x has the midline at 0.50,
    # with the left side (odd joint indices) left of it and the right side right of it.
    # Shoulders are wider than the hips, the arms hang OUTSIDE the hips down to about
    # hip height, and the head sits just above the shoulders, so the figure reads as a
    # real walking body instead of collapsing onto one vertical line.
    xs = {
        0:0.500,                                            # nose
        1:0.485, 2:0.475, 3:0.465, 4:0.515, 5:0.525, 6:0.535,  # eyes (left then right)
        7:0.455, 8:0.545,                                   # ears
        9:0.485, 10:0.515,                                  # mouth
        11:0.415, 12:0.585,                                 # shoulders (wide)
        13:0.395, 14:0.605,                                 # elbows (arms hang outside)
        15:0.405, 16:0.595,                                 # wrists
        17:0.395, 18:0.605, 19:0.405, 20:0.595, 21:0.420, 22:0.580,  # hands track their wrist
        23:0.455, 24:0.545,                                 # hips (narrower than shoulders)
        25:0.450, 26:0.550,                                 # knees
        27:0.448, 28:0.552,                                 # ankles
        29:0.448, 30:0.552, 31:0.455, 32:0.545,             # heels, foot tips
    }
    ys = {
        0:0.16,                                             # nose
        1:0.145, 2:0.145, 3:0.145, 4:0.145, 5:0.145, 6:0.145,  # eyes
        7:0.155, 8:0.155,                                   # ears
        9:0.185, 10:0.185,                                  # mouth (short neck to shoulders)
        11:0.24, 12:0.24,                                   # shoulders
        13:0.38, 14:0.38,                                   # elbows
        15:0.51, 16:0.51,                                   # wrists (about hip height)
        17:0.545, 18:0.545, 19:0.545, 20:0.545, 21:0.535, 22:0.535,  # hands (just past wrists)
        23:0.50, 24:0.50,                                   # hips
        25:0.71, 26:0.71,                                   # knees
        27:0.92, 28:0.92,                                   # ankles
        29:0.94, 30:0.94, 31:0.965, 32:0.965,               # heels, foot tips
    }
    for j in range(33):
        base[j, 0] = xs[j]
        base[j, 1] = ys[j]
    seq = np.repeat(base[None], T, axis=0)
    t = np.linspace(0, 2*np.pi, T, endpoint=False)
    swing = 0.07 * np.sin(t)
    for k, amp in [(25,1.0),(27,1.3),(31,1.4),(13,-0.8),(15,-1.0)]:
        seq[:, k, 0] += swing * amp
    for k, amp in [(26,-1.0),(28,-1.3),(32,-1.4),(14,0.8),(16,1.0)]:
        seq[:, k, 0] += swing * amp
    seq += rng.randn(T, 33, 3).astype(np.float32) * 0.004
    return seq

def animate_skeleton(seq, edges, title="Walking skeleton", fps=8):
    """Animate a (T, 33, C) skeleton inline (uses x=seq[...,0], y=seq[...,1])."""
    T = seq.shape[0]; x_all = seq[:, :, 0]; y_all = seq[:, :, 1]
    groups = [list(range(11)), [11,13,15,17,19,21], [12,14,16,18,20,22],
              [11,12,23,24], [23,25,27,29,31], [24,26,28,30,32]]
    colors = ["#8b5cf6","#3b82f6","#ef4444","#22c55e","#f59e0b","#ec4899"]
    fig, ax = plt.subplots(figsize=(6, 7))
    x_min, x_max = x_all.min(), x_all.max(); y_min, y_max = y_all.min(), y_all.max()
    margin = max(x_max - x_min, y_max - y_min) * 0.1 + 1e-3
    def draw_frame(t):
        ax.clear(); ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
        ax.set_xlim(x_min - margin, x_max + margin); ax.set_ylim(y_max + margin, y_min - margin)
        ax.set_title(f"{title} (frame {t}/{T})")
        x = x_all[t]; y = y_all[t]
        for g_idx, grp in enumerate(groups):
            for (i, j) in [(i, j) for (i, j) in edges if i in grp and j in grp]:
                ax.plot([x[i], x[j]], [y[i], y[j]], color=colors[g_idx], linewidth=2, alpha=0.7)
            ax.scatter([x[i] for i in grp], [y[i] for i in grp], c=colors[g_idx],
                       s=40, zorder=3, edgecolors='white', linewidths=0.5)
    anim = FuncAnimation(fig, draw_frame, frames=T, interval=1000/fps, repeat=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

print("animate_skeleton helper defined.")

In [ ]:
demo_skeleton = synthesize_walking_skeleton(T=16, seed=1)
print("A synthetic walking skeleton: what we are collecting all these videos for.")
display(animate_skeleton(demo_skeleton, EDGES, title="A walking sequence"))

## Recap and what comes next

We collapsed the manifest to its unique videos, downloaded each one just once with
a resumable, fallback-friendly function, and saved a report of what landed on
disk. No pose estimation has happened yet; we only have raw video files.

In notebook 02 we open each cached video, use the bounding boxes from the CSVs to
focus on the walking person, and run MediaPipe BLAZEPOSE_33 to turn every frame
into 33 tracked joints. That is where raw pixels finally become the skeleton
sequences the JEPA learns from.